# Pattern 8: Full Governance Stack (All Layers Together)

The complete security model, combining every pattern from this series into a single
gateway. This is the multi-tenant production shape: each user authenticates with Cognito,
Cedar authorizes them at the gateway, and a Lambda interceptor reads their JWT `department`
claim and injects a metadata filter — so **two users hitting the same gateway, the same tool,
and the same knowledge base see completely different documents**, scoped to their department.

That last part is the payoff the earlier patterns pointed at but never showed end-to-end:
Pattern 5 authenticated the user, Pattern 6 authorized them, Pattern 7 ran an interceptor —
here the interceptor turns *who the user is* (a JWT claim) into *what they can see* (a
document filter), tying the identity layer to the data layer.

## The layers, and what each one independently guarantees

```
Agent ─JWT─► [1] Cognito ─► [2] Gateway ─► [3] Cedar ─► [4] Lambda (REQUEST) ─► [5] KB + filter
             validates       hides KB ID   permit/      reads JWT dept claim,     returns only
             the token                     deny         injects metadata filter   matching docs
```

| Layer | Component | Question it answers | Failure mode |
|---|---|---|---|
| 1 | Cognito JWT | Is this a real, authenticated user? | 401/403 at the edge |
| 2 | Gateway | (structural) Is the KB ID hidden behind one MCP endpoint? | KB ID never exposed |
| 3 | Cedar Policy Engine | Is this principal allowed to invoke this gateway? | 403 (ENFORCE) / logged (LOG_ONLY) |
| 4 | Lambda interceptor | What filter applies to *this* caller? | injects per-department filter |
| 5 | Metadata filter | Which documents can they see? | non-matching docs are dropped |

The design property: **no single layer's failure grants unauthorized access.** If Cedar is too
permissive, the interceptor still scopes the data. If the interceptor doesn't fire, the caller
still had to pass JWT + Cedar to get here — through a gateway that never revealed the KB ID.

## Prerequisites

- Run [Pattern 1](01-direct-sdk.ipynb) first — it creates the shared bucket, uploads the sample
  documents **with department metadata sidecars**, and creates the KB execution role. (The setup
  cell here re-runs it idempotently and additionally creates the Gateway role.)
- IAM permissions for Bedrock, AgentCore (`bedrock-agentcore-control` — Gateway, Policy Engine,
  Cedar), Cognito, Lambda, S3, and IAM.

## The data this notebook scopes on

The two sample docs are tagged (via `util.SAMPLE_FILE_METADATA`, uploaded as `.metadata.json`
sidecars) with a `department` attribute:

| Document | `department` |
|---|---|
| `octank_financial_10K.pdf` (10-K financials) | `finance` |
| `tornadoes_report.pdf` (NOAA tornado report) | `operations` |

So a `finance` user should only ever retrieve Octank financial content, and an `operations`
user only tornado content — no matter what either one asks.

In [ ]:
import boto3
import time
import json
import zipfile
import io
import util   # util.py in this folder — shared bucket + upload + roles

# --- Configuration ---
REGION = "us-west-2"
S3_BUCKET = "<existing-or-unique-name-for-your-kb-bucket->"
S3_PREFIX = "documents/"

session = boto3.Session()

# Reuse the SAME bucket + docs + KB execution role as Pattern 1 (idempotent). The
# upload tags each doc with a `department` metadata sidecar (finance / operations)
# — that is what Layer 5 filters on. Then create the Gateway role the AgentCore
# Gateway assumes to retrieve.
info = util.setup(
    bucket_name=S3_BUCKET,
    prefix=S3_PREFIX,
    metadata=util.SAMPLE_FILE_METADATA,   # department + access_level sidecars
    region_name=REGION,
)
ROLE_ARN     = info["role_arn"]
S3_BUCKET    = info["bucket"]
S3_PREFIX    = info["prefix"]
GW_ROLE_ARN  = util.create_gateway_role(region_name=REGION)
GW_ROLE_NAME = GW_ROLE_ARN.split("/")[-1]

# Clients
cp = session.client("bedrock-agent", region_name=REGION)
dp = session.client("bedrock-agent-runtime", region_name=REGION)
ac = session.client("bedrock-agentcore-control", region_name=REGION)
cognito = session.client("cognito-idp", region_name=REGION)
lambda_client = session.client("lambda", region_name=REGION)
iam = session.client("iam")
ACCOUNT_ID = session.client("sts").get_caller_identity()["Account"]
S3_ACCOUNT = ACCOUNT_ID

print(f"boto3 {boto3.__version__}")
print(f"KB role:      {ROLE_ARN}")
print(f"Gateway role: {GW_ROLE_ARN}")
print(f"Doc metadata: {util.SAMPLE_FILE_METADATA}")

In [ ]:
# Step 1 (Layer 1): Cognito User Pool + two users in different departments.
# The `custom:department` attribute is what the interceptor (Layer 4) reads from
# the ID token to decide which documents each user may retrieve.
pool = cognito.create_user_pool(
    PoolName=f"p8-full-{int(time.time())}",
    Policies={"PasswordPolicy": {
        "MinimumLength": 8, "RequireUppercase": False,
        "RequireLowercase": False, "RequireNumbers": False, "RequireSymbols": False
    }},
    Schema=[{
        "Name": "department", "AttributeDataType": "String",
        "Mutable": True, "Required": False,
        "StringAttributeConstraints": {"MinLength": "1", "MaxLength": "256"}
    }]
)
pool_id = pool["UserPool"]["Id"]

client_resp = cognito.create_user_pool_client(
    UserPoolId=pool_id, ClientName="p8-client",
    ExplicitAuthFlows=["ALLOW_USER_PASSWORD_AUTH", "ALLOW_REFRESH_TOKEN_AUTH"],
    GenerateSecret=False
)
client_id = client_resp["UserPoolClient"]["ClientId"]
discovery_url = f"https://cognito-idp.{REGION}.amazonaws.com/{pool_id}/.well-known/openid-configuration"
print(f"Cognito Pool: {pool_id}")
print(f"App Client:   {client_id}")

# Two users whose departments match the document metadata tags (Step 0 table):
#   fin-user  -> department=finance     (should see only Octank 10-K content)
#   ops-user  -> department=operations  (should see only tornado-report content)
for username, dept in [("fin-user", "finance"), ("ops-user", "operations")]:
    cognito.admin_create_user(
        UserPoolId=pool_id, Username=username,
        UserAttributes=[
            {"Name": "custom:department", "Value": dept},
            {"Name": "email", "Value": f"{username}@example.com"}
        ],
        MessageAction="SUPPRESS"
    )
    cognito.admin_set_user_password(
        UserPoolId=pool_id, Username=username,
        Password="TestPass1", Permanent=True
    )
    print(f"  User: {username} (department={dept})")

In [ ]:
# Step 2 (Layer 3): Create the Cedar Policy Engine.
# Cedar answers "is this principal allowed to invoke this gateway at all?" We
# attach it to the gateway in Step 5 and add a permit policy in Step 6.
pe = ac.create_policy_engine(name=f"p8pe{int(time.time())}")
pe_id = pe["policyEngineId"]
pe_arn = pe["policyEngineArn"]
for _ in range(12):
    if ac.get_policy_engine(policyEngineId=pe_id)["status"] == "ACTIVE":
        break
    time.sleep(5)
print(f"Policy Engine: {pe_id} ACTIVE")

In [ ]:
# Step 3: Create KB + Data Source + Ingest (same as Pattern 1).
# The department metadata sidecars uploaded in the setup cell are indexed during
# this ingestion (SMART_PARSING), which is what makes the Layer 5 filter work.
response = cp.create_knowledge_base(
    name=f"p8-full-{int(time.time())}",
    roleArn=ROLE_ARN,
    knowledgeBaseConfiguration={
        "type": "MANAGED",
        "managedKnowledgeBaseConfiguration": {}   # empty = managed default embedding
    }
)
kb_id = response["knowledgeBase"]["knowledgeBaseId"]
print(f"KB: {kb_id}")

for _ in range(30):
    if cp.get_knowledge_base(knowledgeBaseId=kb_id)["knowledgeBase"]["status"] == "ACTIVE":
        break
    time.sleep(5)
print("KB ACTIVE")

response = cp.create_data_source(
    knowledgeBaseId=kb_id,
    name="s3-source",
    dataSourceConfiguration={
        "type": "MANAGED_KNOWLEDGE_BASE_CONNECTOR",
        "managedKnowledgeBaseConnectorConfiguration": {
            "connectorParameters": {
                "type": "S3",
                "version": "1",
                "connectionConfiguration": {
                    "bucketName": S3_BUCKET,
                    "bucketOwnerAccountId": S3_ACCOUNT
                },
                "filterConfiguration": {"inclusionPrefixes": [S3_PREFIX]},
                "deletionProtectionConfiguration": {"enableDeletionProtection": False}
            },
            "deletionProtectionConfiguration": {"deletionProtectionStatus": "DISABLED"}
        }
    },
    vectorIngestionConfiguration={
        "parsingConfiguration": {"parsingStrategy": "SMART_PARSING"}
    }
)
ds_id = response["dataSource"]["dataSourceId"]
print(f"DS: {ds_id}")

for _ in range(12):
    if cp.get_data_source(knowledgeBaseId=kb_id, dataSourceId=ds_id)["dataSource"]["status"] == "AVAILABLE":
        break
    time.sleep(5)
print("DS AVAILABLE")

response = cp.start_ingestion_job(knowledgeBaseId=kb_id, dataSourceId=ds_id)
job_id = response["ingestionJob"]["ingestionJobId"]
for _ in range(40):
    job = cp.get_ingestion_job(
        knowledgeBaseId=kb_id, dataSourceId=ds_id, ingestionJobId=job_id
    )["ingestionJob"]
    if job["status"] in ("COMPLETE", "FAILED"):
        break
    time.sleep(15)
print(f"Ingestion: {job['status']}")

In [ ]:
# Step 4 (Layer 4, part 1): Lambda execution role (basic CloudWatch Logs access).
# Reused across re-runs — same role as Pattern 7's interceptor.
lambda_role_name = "AmazonBedrockInterceptorLambdaRole"
trust = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "lambda.amazonaws.com"},
        "Action": "sts:AssumeRole",
    }],
}
try:
    r = iam.create_role(RoleName=lambda_role_name, AssumeRolePolicyDocument=json.dumps(trust))
    iam.attach_role_policy(
        RoleName=lambda_role_name,
        PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole",
    )
    LAMBDA_ROLE_ARN = r["Role"]["Arn"]
    print(f"Created Lambda role: {lambda_role_name}")
    print("  Waiting 12s for IAM propagation...")
    time.sleep(12)
except iam.exceptions.EntityAlreadyExistsException:
    LAMBDA_ROLE_ARN = iam.get_role(RoleName=lambda_role_name)["Role"]["Arn"]
    print(f"Lambda role already exists: {lambda_role_name}")

print(f"LAMBDA_ROLE_ARN = {LAMBDA_ROLE_ARN}")

In [ ]:
# Step 4 (Layer 4, part 2): The interceptor Lambda — the layer that ties identity to data.
#
# It runs as a REQUEST interceptor (before the KB is called). On every tools/call it:
#   1. reads the caller's ID token from the `x-auth-token` header (forwarded because
#      the gateway is created with passRequestHeaders=True),
#   2. decodes the `custom:department` claim, and
#   3. injects a metadata filter into the tool-call arguments so the KB returns ONLY
#      that department's documents.
#
# The gateway already verified the caller's ACCESS token at the edge (Layer 1), so the
# ID token we decode here is trustworthy — we read its claims WITHOUT re-verifying the
# signature. The agent never sets (or knows) the filter; scoping is enforced server-side.
#
# IMPORTANT: the KB connector rejects an injected `retrievalConfiguration` UNLESS the
# target explicitly exposes that path via `parameterOverrides` (Step 7). Without it the
# call fails with "IllegalArgumentException ... cannot set parameter(s): [/retrievalConfiguration]".
handler_code = r'''
import json, logging, base64

logger = logging.getLogger()
logger.setLevel(logging.INFO)


def _decode_jwt_claims(token):
    """Decode a JWT payload WITHOUT verifying the signature.

    Safe here because the gateway already validated the token at the edge before
    invoking this interceptor. Never do this for a token the gateway hasn't verified.
    """
    payload = token.split(".")[1]
    payload += "=" * (-len(payload) % 4)   # restore base64 padding
    return json.loads(base64.urlsafe_b64decode(payload))


def handler(event, context):
    mcp = event.get("mcp", {})
    gw_request = mcp.get("gatewayRequest", {})
    body = gw_request.get("body", {}) or {}
    headers = gw_request.get("headers", {}) or {}
    # Header names are case-insensitive; normalize before lookup.
    hdr = {k.lower(): v for k, v in headers.items()}

    # Pass non-tool-call methods (initialize, tools/list, ...) straight through.
    passthrough = {
        "interceptorOutputVersion": "1.0",
        "mcp": {"transformedGatewayRequest": {"body": body}},
    }
    if body.get("method") != "tools/call":
        return passthrough

    auth = hdr.get("x-auth-token", "")
    if auth.startswith("Bearer "):
        try:
            claims = _decode_jwt_claims(auth[len("Bearer "):])
            department = claims.get("custom:department")
            if department:
                # Inject the per-user metadata filter into the tool-call arguments.
                args = body.setdefault("params", {}).setdefault("arguments", {})
                args["retrievalConfiguration"] = {
                    "managedSearchConfiguration": {
                        "filter": {"equals": {"key": "department", "value": department}}
                    }
                }
                logger.info(f"AUDIT user={claims.get('sub')} department={department} "
                            f"-> injected filter department=={department}")
        except Exception as e:
            logger.error(f"JWT decode/inject failed: {e}")

    # Forward the (now filtered) request to the KB target.
    return {
        "interceptorOutputVersion": "1.0",
        "mcp": {"transformedGatewayRequest": {"body": body}},
    }
'''

# Package + create the function
zip_buffer = io.BytesIO()
with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.writestr("lambda_function.py", handler_code)
zip_buffer.seek(0)

func_name = f"p8-interceptor-{int(time.time())}"
fn = lambda_client.create_function(
    FunctionName=func_name,
    Runtime="python3.12",
    Role=LAMBDA_ROLE_ARN,
    Handler="lambda_function.handler",
    Code={"ZipFile": zip_buffer.read()},
    Timeout=30
)
LAMBDA_ARN = fn["FunctionArn"]
print(f"Lambda: {LAMBDA_ARN}")

for _ in range(12):
    state = lambda_client.get_function(FunctionName=func_name)["Configuration"]["State"]
    if state == "Active":
        break
    time.sleep(5)
print(f"Lambda state: {state}")

In [ ]:
# Step 4 (Layer 4, part 3): Let the GATEWAY SERVICE ROLE invoke the interceptor.
# The gateway calls the interceptor Lambda using its own service role, so that role
# needs lambda:InvokeFunction on the function. Without this, tool calls fail with an
# HTTP 500 BEFORE the interceptor ever runs (there won't even be a Lambda log group).
iam.put_role_policy(
    RoleName=GW_ROLE_NAME,
    PolicyName="InterceptorInvoke",
    PolicyDocument=json.dumps({
        "Version": "2012-10-17",
        "Statement": [{
            "Effect": "Allow",
            "Action": ["lambda:InvokeFunction"],
            "Resource": [LAMBDA_ARN],
        }],
    }),
)
print(f"Granted {GW_ROLE_NAME} lambda:InvokeFunction on the interceptor")
print("  Waiting 12s for IAM propagation...")
time.sleep(12)

In [ ]:
# Step 5 (Layers 1+2+3+4 on one gateway): create the Gateway with EVERYTHING.
#   - authorizerType=CUSTOM_JWT           -> Layer 1 (Cognito validates the token)
#   - policyEngineConfiguration           -> Layer 3 (Cedar; LOG_ONLY to start)
#   - interceptorConfigurations (REQUEST) -> Layer 4 (our filter-injecting Lambda)
#   - passRequestHeaders=True             -> so the interceptor can read x-auth-token
#
# On allowedClients (and NOT allowedAudience): the authorizer verifies EVERY field you
# configure, but Cognito splits these claims across its two tokens — the ACCESS token
# carries `client_id` (no `aud`); the ID token carries `aud` (no `client_id`). We present
# the access token for gateway auth, so requiring `allowedAudience` too would 403 it.
gw_response = ac.create_gateway(
    name=f"p8-full-gw-{int(time.time())}",
    roleArn=GW_ROLE_ARN,
    protocolType="MCP",
    authorizerType="CUSTOM_JWT",
    authorizerConfiguration={"customJWTAuthorizer": {
        "discoveryUrl": discovery_url,
        "allowedClients": [client_id]
    }},
    policyEngineConfiguration={"arn": pe_arn, "mode": "LOG_ONLY"},
    interceptorConfigurations=[{
        "interceptor": {"lambda": {"arn": LAMBDA_ARN}},
        "interceptionPoints": ["REQUEST"],
        "inputConfiguration": {"passRequestHeaders": True}
    }]
)

gw_id = gw_response["gatewayId"]
print(f"Gateway: {gw_id}")

gw_url = None
for _ in range(24):
    gw = ac.get_gateway(gatewayIdentifier=gw_id)
    if gw["status"] == "READY":
        gw_url = gw.get("gatewayUrl", "N/A")
        break
    elif "FAIL" in gw["status"]:
        print(f"FAILED: {gw.get('statusReasons', [])}")
        break
    time.sleep(5)
print(f"Status: {gw['status']}")
print(f"Gateway URL: {gw_url}")

In [ ]:
# Step 6 (Layer 3): Cedar policy — permit callers to invoke this gateway.
# LOG_ONLY mode means Cedar evaluates + logs this decision but does not block; in
# production you would scope the permit to specific principals and switch to ENFORCE
# (see Pattern 4). Policy names must be unique per engine, so derive one from gw_id.
gw_arn = f"arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT_ID}:gateway/{gw_id}"
cedar_statement = f'permit(principal, action, resource == AgentCore::Gateway::"{gw_arn}");'
print(f"Cedar policy:\n  {cedar_statement}")

policy_response = ac.create_policy(
    policyEngineId=pe_id,
    name=f"permit_gateway_{gw_id.split('-')[-1]}",
    definition={"cedar": {"statement": cedar_statement}},
    validationMode="IGNORE_ALL_FINDINGS"
)
policy_id = policy_response["policyId"]

for _ in range(12):
    p = ac.get_policy(policyEngineId=pe_id, policyId=policy_id)
    if p["status"] == "ACTIVE":
        break
    time.sleep(5)
print(f"Policy: {policy_id} [{p['status']}]")

In [ ]:
# Step 7 (Layer 5): KB Target that EXPOSES the filter path for runtime injection.
#
# The `parameterValues` bind the KB ID and defaults (never sent by the caller). The
# crucial piece is `parameterOverrides`: it publishes the filter path into the tool
# schema so the interceptor may set it at runtime. WITHOUT this block, the connector
# rejects any interceptor-injected retrievalConfiguration with:
#   "IllegalArgumentException - Invalid request parameters: cannot set parameter(s): [/retrievalConfiguration]"
# WITH it, the connector merges the injected filter into the target's default config.
target_response = ac.create_gateway_target(
    gatewayIdentifier=gw_id,
    name="kb-retrieve",
    targetConfiguration={
        "mcp": {
            "connector": {
                "source": {"connectorId": "bedrock-knowledge-bases"},
                "configurations": [{
                    "name": "Retrieve",
                    "description": (
                        "Search corporate documents: Octank Financial's 10-K annual report "
                        "(finance) and a U.S. tornado background & forecasting report (operations). "
                        "Results are automatically scoped to the caller's department."
                    ),
                    "parameterValues": {
                        "knowledgeBaseId": kb_id,
                        "retrievalConfiguration": {
                            "managedSearchConfiguration": {"numberOfResults": 5}
                        }
                    },
                    # Expose the metadata-filter path so the interceptor can inject it.
                    "parameterOverrides": [{
                        "path": "$.retrievalConfiguration.managedSearchConfiguration.filter",
                        "description": "Metadata filter injected per-caller by the interceptor",
                        "visible": True
                    }]
                }]
            }
        }
    },
    credentialProviderConfigurations=[
        {"credentialProviderType": "GATEWAY_IAM_ROLE"}
    ]
)

target_id = target_response["targetId"]
print(f"Target: {target_id}")

for _ in range(12):
    t = ac.get_gateway_target(gatewayIdentifier=gw_id, targetId=target_id)
    if t["status"] == "READY":
        break
    time.sleep(5)
print(f"Target status: {t['status']}")
print("Waiting 15s for the interceptor + target wiring to settle...")
time.sleep(15)

In [ ]:
# Step 8: Helpers — log a user in, and call the gateway on their behalf.
#
# Cognito issues two tokens; we use them for two different jobs:
#   - ACCESS token -> Authorization header. The gateway validates this at the edge
#     (it carries the `client_id` the authorizer checks). It has NO custom claims.
#   - ID token     -> x-auth-token header. It carries `custom:department`, which the
#     interceptor decodes to build the filter. (In a real deployment the agent is
#     simply handed a pre-issued token; it never sets or sees the filter.)
import httpx
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client


def login(username, password="TestPass1"):
    """Return (access_token, id_token) for a Cognito user."""
    auth = cognito.initiate_auth(
        ClientId=client_id,
        AuthFlow="USER_PASSWORD_AUTH",
        AuthParameters={"USERNAME": username, "PASSWORD": password},
    )["AuthenticationResult"]
    return auth["AccessToken"], auth["IdToken"]


async def gateway_retrieve(username, query_text):
    """Authenticate as `username`, then retrieve through the full-governance gateway.

    The department filter is NOT passed here — the interceptor injects it from the
    caller's JWT. Retries transient ConnectTimeouts a freshly-created gateway can throw.
    """
    access_token, id_token = login(username)
    headers = {
        "Authorization": f"Bearer {access_token}",   # Layer 1 — gateway edge auth
        "x-auth-token": f"Bearer {id_token}",         # Layer 4 — read by interceptor
    }
    last_err = None
    for _ in range(5):
        try:
            async with httpx.AsyncClient(headers=headers, timeout=60) as http_client:
                async with streamable_http_client(gw_url, http_client=http_client) as (read, write, _):
                    async with ClientSession(read, write) as session_mcp:
                        await session_mcp.initialize()
                        listing = await session_mcp.list_tools()
                        tool_name = next(t.name for t in listing.tools
                                         if t.name.split("___")[-1] == "Retrieve")
                        return await session_mcp.call_tool(
                            name=tool_name,
                            arguments={"retrievalQuery": {"text": query_text}},
                        )
        except Exception as e:
            last_err = e
            await asyncio.sleep(6)
    raise last_err


import asyncio
print("Helpers ready: login(), gateway_retrieve()")

In [ ]:
# Step 9: The payoff — SAME gateway, SAME tool, SAME KB, DIFFERENT documents per user.
# Each user asks a question that spans BOTH documents ("summarize the key facts"), so
# the only thing deciding what comes back is the department filter the interceptor
# injected from their JWT. finance -> Octank 10-K only; operations -> tornado report only.
QUERY = "Summarize the key facts in the available documents."

# Distinctive tokens unique to each document (no cross-matches in the other doc's text).
FINANCE_MARKERS = ("octank", "stockholders", "balance sheet", "10-k")
OPERATIONS_MARKERS = ("tornado", "noaa", "thunderstorm")

def summarize(username, result):
    text = result.content[0].text if result.content else ""
    low = text.lower()
    finance_hit = any(k in low for k in FINANCE_MARKERS)
    ops_hit = any(k in low for k in OPERATIONS_MARKERS)
    print(f"[{username}] isError={result.isError} | finance docs={finance_hit} | operations docs={ops_hit}")
    print(f"   {text[:200].strip()}...\n")

print(f"Query (identical for both users): {QUERY!r}\n")

fin_result = await gateway_retrieve("fin-user", QUERY)
summarize("fin-user (department=finance)", fin_result)

ops_result = await gateway_retrieve("ops-user", QUERY)
summarize("ops-user (department=operations)", ops_result)

print("Same endpoint, same tool, same KB — the JWT department claim (via the interceptor)")
print("is the ONLY thing that scoped each user to different documents.")

## What just happened, end to end

For each `gateway_retrieve(user, query)` call:

1. **Cognito (Layer 1)** — the client logs in and receives an access token + an ID token.
2. **Gateway edge (Layers 1+2)** — the request arrives at one MCP endpoint with the access
   token in `Authorization`. The gateway validates signature/expiry/`client_id` via OIDC
   discovery. The caller never sees the KB ID.
3. **Cedar (Layer 3)** — the policy engine evaluates the permit for this gateway (LOG_ONLY
   here, so it logs the decision to CloudWatch).
4. **Lambda interceptor (Layer 4)** — runs before the KB call, decodes `custom:department`
   from the ID token in `x-auth-token`, and injects
   `retrievalConfiguration.managedSearchConfiguration.filter = {equals: department == <dept>}`.
5. **KB + metadata filter (Layer 5)** — the connector merges the injected filter into the
   target's default config (allowed because Step 7 exposed that path via `parameterOverrides`)
   and returns only the caller's department's chunks.

Result: two identical queries, two disjoint document sets — scoped entirely by identity.

## Why this is defense-in-depth

| Layer | Question answered | If it alone failed… |
|---|---|---|
| JWT (Cognito) | Is this a valid, authenticated user? | …the request still hit Cedar + a hidden KB ID |
| Cedar | Is this principal allowed on this gateway? | …the interceptor still scoped the data |
| Interceptor | What filter applies to this caller? | …the caller still passed JWT + Cedar first |
| Metadata filter | Which documents can they see? | …the KB ID was still never exposed |
| Gateway | Is the KB ID hidden? | …callers still needed a valid JWT to reach it |

No single layer's failure grants unauthorized access to the wrong documents.

## Extending Layer 4: RESPONSE interceptors (audit / redaction)

This notebook uses a **REQUEST** interceptor because that is where filter injection belongs
(before the KB is queried). The same Lambda mechanism supports a **RESPONSE** interceptor for
audit logging or PII redaction — attach it by adding `"RESPONSE"` to `interceptionPoints`.

One caveat specific to the managed-KB connector: it **streams** its response
(`isStreamingResponse: true`), so a RESPONSE interceptor is invoked once per streamed event
rather than once with the full body, and it must follow the streaming output contract
(only the first event may override `headers`/`statusCode`; subsequent events may override
`body` only). See
[Response interceptors with streaming enabled](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway-interceptors-types.html).
For simple buffered audit/redaction, an HTTP target (AgentCore Runtime) is the more
straightforward fit.

## When to use this pattern

- Multi-tenant SaaS with per-tenant / per-department document isolation
- Regulated industries requiring an audit trail and identity-scoped retrieval
- Any deployment where different roles must see different document sets from one KB

In [ ]:
# Cleanup — order: policy → target → gateway → policy engine → Lambda → Cognito → DS → KB
ac.delete_policy(policyEngineId=pe_id, policyId=policy_id)
ac.delete_gateway_target(gatewayIdentifier=gw_id, targetId=target_id)
time.sleep(3)
ac.delete_gateway(gatewayIdentifier=gw_id)
time.sleep(3)
ac.delete_policy_engine(policyEngineId=pe_id)
lambda_client.delete_function(FunctionName=LAMBDA_ARN)
cognito.delete_user_pool(UserPoolId=pool_id)
cp.delete_data_source(knowledgeBaseId=kb_id, dataSourceId=ds_id)
cp.delete_knowledge_base(knowledgeBaseId=kb_id)
print(f"Deleted: Policy {policy_id}, Gateway {gw_id}, PE {pe_id}")
print(f"Deleted: Lambda {LAMBDA_ARN}, Cognito {pool_id}, KB {kb_id}")

# We intentionally DO NOT delete two reusable artifacts, so re-runs stay idempotent and a
# concurrent run can't strip the invoke permission from a live gateway (same as Pattern 7):
#   - the InterceptorInvoke inline policy on GW_ROLE_NAME (Step 4, part 3)
#   - the Lambda execution role AmazonBedrockInterceptorLambdaRole (Step 4, part 1)
# To tear the whole pattern down for good, delete them manually:
#   iam.delete_role_policy(RoleName=GW_ROLE_NAME, PolicyName="InterceptorInvoke")
#   # (detach AWSLambdaBasicExecutionRole, then) iam.delete_role(RoleName="AmazonBedrockInterceptorLambdaRole")